In [1]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.models.lite_llm import LiteLlm
from google.genai.types import Content,Part

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries Imported")

# Provide Required API Keys

In [ ]:
os.environ["GOOGLE_API_KEY"] = '<...>'
os.environ['OPENAI_API_KEY'] = '<...>'
os.environ['ANTHROPIC_API_KEY'] = '<...>'

In [ ]:
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash-exp"
MODEL_GPT_4O = "openai/gpt-4o"
MODEL_CLAUDE_SONNET = "anthropic/claude-3-sonnet-20240229"


print("\nEnvironment configured.")

In [ ]:
# @title Define the get_weather Tool
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    # Best Practice: Log tool execution for easier debugging
    print(f"--- Tool: get_weather called for city: {city} ---")
    city_normalized = city.lower().replace(" ", "") # Basic input normalization

    # Mock weather data for simplicity
    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    # Best Practice: Handle potential errors gracefully within the tool
    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

# Example tool usage (optional self-test)
print(get_weather("New York"))
print(get_weather("Paris"))

### Define Agent

In [ ]:
AGENT_MODEL = MODEL_GEMINI_2_0_FLASH

weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL,
    description="""Provide weather information for specific cities""",
    instruction="""
    You are a helpful weather assistant. Your primary goal is to provide current weather reports.
    When the user asks for the weather in a specific city,
    you MUST use the 'get_weather' tool to find the information. 
    Analyze the tool's response: if the status is 'error', inform the user politely about the error message. 
    If the status is 'success', present the weather 'report' clearly and concisely to the user. 
    Only use the tool when a city is mentioned for a weather request.
    """,
    tools=[get_weather],
)

print(f"Agent {weather_agent.name} created using model {AGENT_MODEL}")

### Setup Session Service and Runner

In [ ]:
# SessionService stores conversation history and state
session_service = InMemorySessionService()



APP_NAME = "weather_tutorial_app"
USER_ID = "user_1"
SESSION_ID = "session_001" 

session = session_service.create_session(
    app_name=APP_NAME,
    session_id=SESSION_ID,
    user_id=USER_ID,
)

print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# Runner orchestrate the agent execution loop
runner = Runner(
    app_name=APP_NAME,
    agent=weather_agent,
    session_service=session_service,
)
print(f"Runner created for agent '{runner.agent.name}'.")


In [8]:
async def call_agent_async(query:str,runner,user_id,session_id):
    """Sends a query to the agent and print the final response"""
    print(f"\n>>>User Query: {query}")

    content = Content(role="user",parts=[Part(text=query)])

    final_response_text = "Agent did not produce a final response."

    async for event in runner.run_async(user_id=user_id,session_id=session_id,new_message=content):
        # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                final_response_text = f"Agent escalated: {event.error_message or 'No specific message'}"
                break
    print(f"<<< Agent Response: {final_response_text}")


In [ ]:
async def run_conversation():
    await call_agent_async("What is the weather in london",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)
    await call_agent_async("How about Paris?",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)
    await call_agent_async("Tell me the weather in new york",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)
await run_conversation()

### Define and test Multi-Model Agents

In [ ]:
weather_agent_gpt = None
runner_gpt = None

try:
    weather_agent_gpt = Agent(
        name="weather_agent_gpt",
        model=LiteLlm(model=MODEL_GPT_4O),
        description="Provides weather information (using GPT-4o).",
        instruction="""
                    You are a helpful weather assistant. Your primary goal is to provide current weather reports.
                    When the user asks for the weather in a specific city,
                    you MUST use the 'get_weather' tool to find the information. 
                    Analyze the tool's response: if the status is 'error', inform the user politely about the error message. 
                    If the status is 'success', present the weather 'report' clearly and concisely to the user. 
                    Only use the tool when a city is mentioned for a weather request.
                    """,
        tools=[get_weather],
    )
    print(f"Agent: {weather_agent_gpt.name} created using model {MODEL_GPT_4O}")

    session_service_gpt = InMemorySessionService()

    APP_NAME_GPT = "weather_tutorial_app_gpt"
    USER_ID_GPT = "user_1_gpt"
    SESSION_ID_GPT = "session_001_gpt"

    session_gpt = session_service_gpt.create_session(
        app_name=APP_NAME_GPT,
        session_id=SESSION_ID_GPT,
        user_id=USER_ID_GPT,
    )

    print(f"Session created: App='{APP_NAME_GPT}', User='{USER_ID_GPT}', Session='{SESSION_ID_GPT}'")

    runner_gpt = Runner(
        app_name=APP_NAME_GPT,
        agent=weather_agent_gpt,
        session_service=session_service_gpt,
    )
    print(f"Runner created for agent {runner_gpt.agent.name}")

    # --- Test GPT Agent ---
    print("\n--- Testing GPT Agent ---")
    await call_agent_async(query = "What's the weather in Tokyo?",
                           runner=runner_gpt,
                           user_id=USER_ID_GPT,
                           session_id=SESSION_ID_GPT)
    
except Exception as e:
    print(f"❌ Could not create or run GPT agent '{MODEL_GPT_4O}'. Check API Key and model name. Error: {e}")

    


### Agent Team

In [ ]:
def say_hello(name: str = "there") -> str:
    """Provides a simple greeting, optionally addressing the user by name.

    Args:
        name (str, optional): The name of the person to greet. Defaults to "there".

    Returns:
        str: A friendly greeting message.
    """
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"

def say_goodbye() -> str:
    """Provides a simple farewell message to conclude the conversation."""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

print("Greeting and Farewell tools defined.")

print(say_hello("Alice"))
print(say_goodbye())

In [ ]:
greeting_agent = None
try:
    greeting_agent = Agent(
        name="greeting_agent",
        model=LiteLlm(model=MODEL_GPT_4O),
        description="""Handles simple greetings and hellos using the 'say_hello' tool """,
        instruction="""
                    You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user.
                    Use the 'say_hello' tool to generate the greeting.
                    If the user provides their name, make sure to pass it to the tool.
                    Do not engage in any other conversation or tasks.
                    """,
        tools=[say_hello],
        )
    print(f"Agent: {greeting_agent.name} created using model {greeting_agent.model}")
except Exception as e:
    print(f"Could note create Greeting Agent. Error {e}")

In [ ]:
farewell_agent = None
try:
    farewell_agent = Agent(
        name="farewell_agent",
        model=MODEL_GEMINI_2_0_FLASH,
        description="""Handles simple farewells and goodbyes using the 'say_goodbye' tool.""",
        instruction="""
                    You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message.
                    Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation
                    (e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you').
                    Do not perform any other actions.
                    """,
        tools=[say_goodbye]
    )
    print(f"Agent: {farewell_agent.name} created using model {farewell_agent.model}")
except Exception as e:
    print(f"Could note create Farewell Agent. Error {e}")

In [ ]:
weather_agent_team = None
try:
    if greeting_agent and farewell_agent and 'get_weather' in globals():
        root_agent_model = MODEL_GEMINI_2_0_FLASH
        weather_agent_team = Agent(
            name="weather_agent_v2",
            model=root_agent_model,
            description="""The main coordinator agent. Handles weather requests and delegates greetings/farewells to specialists.""",
            instruction="""
                        You are the main Weather Agent coordinating a team. Your primary responsibility is to provide weather information.
                        Use the 'get_weather' tool ONLY for specific weather requests (e.g., 'weather in London').
                        You have specialized sub-agents:
                        1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these.
                        2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these.
                        Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. If it's a farewell, delegate to 'farewell_agent'.
                        If it's a weather request, handle it yourself using 'get_weather'.
                        For anything else, respond appropriately or state you cannot handle it.
                        """,
            tools=[get_weather],
            sub_agents=[greeting_agent,farewell_agent]
        )
        print(f"Root Agent {weather_agent_team.name} creates using model {weather_agent_team.model}")
    else:
        print("❌ Cannot create root agent because one or more sub-agents failed to initialize or 'get_weather' tool is missing.")
        if not greeting_agent: print(" - Greeting Agent is missing.")
        if not farewell_agent: print(" - Farewell Agent is missing.")
        if 'get_weather' not in globals(): print(" - get_weather function is missing.")
except Exception as e:
    print(f"Could note create Weather Agent Team. Error {e}")

In [15]:
async def run_team_conversation():
    session_service = InMemorySessionService()
    APP_NAME = "weather_tutorial_agent_team"
    USER_ID = "user_1_agent_team"
    SESSION_ID = "session_001_agent_team"

    session = session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

    runner_agent_team = Runner(
        app_name=APP_NAME,
        agent=weather_agent_team,
        session_service=session_service,
    )

    print(f"Runner created for agent '{runner_agent_team.agent.name}'.")

    await call_agent_async(query = "Hello there!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
    await call_agent_async(query = "What is the weather in New York?",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
    await call_agent_async(query = "Thanks, bye!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)


In [ ]:
await run_team_conversation()

### Adding Memory and Personalization with Session State

In [ ]:
session_service_stateful = InMemorySessionService()

SESSION_ID_STATEFUL = "session_state_demo_001"
USER_ID_STATEFUL = "user_state_demo"

initial_state = {
    "user_preference_temperature_unit":"Celsius"
}

session_stateful = session_service_stateful.create_session(
    app_name=APP_NAME,
    user_id=USER_ID_STATEFUL,
    session_id=SESSION_ID_STATEFUL,
    state=initial_state
)

print(f"Session: {SESSION_ID_STATEFUL} created for user {USER_ID_STATEFUL}")

retrieved_session = session_service_stateful.get_session(app_name=APP_NAME,
                                                         user_id=USER_ID_STATEFUL,
                                                         session_id=SESSION_ID_STATEFUL)
print("\n---Initial Session state---")
if retrieved_session:
    print(retrieved_session)
else:
    print("Error: Could not retrieve session.")

In [ ]:
from google.adk.tools.tool_context import ToolContext

def get_weather_stateful(city: str,tool_context: ToolContext) -> dict:
    """Retrieves weather, converts temp unit based on session state."""
    print(f"--- Tool: get_weather_stateful called for {city} ---")
    preferred_unit = tool_context.state.get("user_preference_temperature_unit","Celsius")
    print(f"--- Tool: Reading state 'user_preference_temperature_unit': {preferred_unit} ---")

    city_normalized = city.lower().replace(" ", "")

    # Mock weather data (always stored in Celsius internally)
    mock_weather_db = {
        "newyork": {"temp_c": 25, "condition": "sunny"},
        "london": {"temp_c": 15, "condition": "cloudy"},
        "tokyo": {"temp_c": 18, "condition": "light rain"},
    }

    if city_normalized in mock_weather_db:
        data = mock_weather_db[city_normalized]
        temp_c = data["temp_c"]
        condition = data["condition"]

        # Format temperature based on state preference
        if preferred_unit == "Fahrenheit":
            temp_value = (temp_c * 9/5) + 32 # Calculate Fahrenheit
            temp_unit = "°F"
        else: # Default to Celsius
            temp_value = temp_c
            temp_unit = "°C"

        report = f"The weather in {city.capitalize()} is {condition} with a temperature of {temp_value:.0f}{temp_unit}."
        result = {"status": "success", "report": report}
        print(f"--- Tool: Generated report in {preferred_unit}. Result: {result} ---")

        # Example of writing back to state (optional for this tool)
        tool_context.state["last_city_checked_stateful"] = city
        print(f"--- Tool: Updated state 'last_city_checked_stateful': {city} ---")

        return result
    else:
        # Handle city not found
        error_msg = f"Sorry, I don't have weather information for '{city}'."
        print(f"--- Tool: City '{city}' not found. ---")
        return {"status": "error", "error_message": error_msg}

print("State-aware 'get_weather_stateful' tool defined.")

In [ ]:
greeting_agent = None
try:
    greeting_agent = Agent(
        name="greeting_agent",
        model=MODEL_GEMINI_2_0_FLASH,
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        tools=[say_hello],
    )
    print(f"Agent: {greeting_agent.name} redefined")
except Exception as e:
    print(f"Could note redefine Greeting agent. Error {e}")

farewell_agent = None
try:
    farewell_agent = Agent(
        name="farewell_agent",
        model=MODEL_GEMINI_2_0_FLASH,
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        tools=[say_goodbye]
    )
    print(f"Agent: {farewell_agent.name} redefined")
except Exception as e:
    print(f"Could note redefine Farewell agent. Error {e}")


In [ ]:
try:
    root_agent_stateful = Agent(
        name="weather_agent_v4_stateful",
        model=MODEL_GEMINI_2_0_FLASH,
        description="""Main agent: Provides weather (state-aware unit), delegates greetings/farewells, saves report to state.""",
        instruction="""
                    You are the main Weather Agent. Your job is to provide weather using 'get_weather_stateful'.
                    The tool will format the temperature based on user preference stored in state.
                    Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'.
                    Handle only weather requests, greetings, and farewells.,
                    """,
        tools=[get_weather_stateful],
        sub_agents=[greeting_agent,farewell_agent],
        output_key="last_weather_report"
    )
    print(f"Root Agent '{root_agent_stateful.name}' created using stateful tool and output_key.")

    runner_root_stateful = Runner(
        app_name=APP_NAME,
        agent=root_agent_stateful,
        session_service=session_service_stateful,
    )
    print(f"Runner created for stateful root agent '{runner_root_stateful.agent.name}' using stateful session service.")
except Exception as e:
    print(f"Failed to Stateful Weather Agent. Error {e}")


In [ ]:
if 'runner_root_stateful' in globals() and runner_root_stateful:
  async def run_stateful_conversation():
      print("\n--- Testing State: Temp Unit Conversion & output_key ---")

      # 1. Check weather (Uses initial state: Celsius)
      print("--- Turn 1: Requesting weather in London (expect Celsius) ---")
      await call_agent_async(query= "What's the weather in London?",
                             runner=runner_root_stateful,
                             user_id=USER_ID_STATEFUL,
                             session_id=SESSION_ID_STATEFUL
                            )

      # 2. Manually update state preference to Fahrenheit - DIRECTLY MODIFY STORAGE
      print("\n--- Manually Updating State: Setting unit to Fahrenheit ---")
      try:
          # Access the internal storage directly - THIS IS SPECIFIC TO InMemorySessionService for testing
          stored_session = session_service_stateful.sessions[APP_NAME][USER_ID_STATEFUL][SESSION_ID_STATEFUL]
          stored_session.state["user_preference_temperature_unit"] = "Fahrenheit"
          print(f"--- Stored session state updated. Current 'user_preference_temperature_unit': {stored_session.state['user_preference_temperature_unit']} ---")
      except KeyError:
          print(f"--- Error: Could not retrieve session '{SESSION_ID_STATEFUL}' from internal storage for user '{USER_ID_STATEFUL}' in app '{APP_NAME}' to update state. Check IDs and if session was created. ---")
      except Exception as e:
           print(f"--- Error updating internal session state: {e} ---")

      # 3. Check weather again (Tool should now use Fahrenheit)
      # This will also update 'last_weather_report' via output_key
      print("\n--- Turn 2: Requesting weather in New York (expect Fahrenheit) ---")
      await call_agent_async(query= "Tell me the weather in New York.",
                             runner=runner_root_stateful,
                             user_id=USER_ID_STATEFUL,
                             session_id=SESSION_ID_STATEFUL
                            )

      # 4. Test basic delegation (should still work)
      # This will update 'last_weather_report' again, overwriting the NY weather report
      print("\n--- Turn 3: Sending a greeting ---")
      await call_agent_async(query= "Hi!",
                             runner=runner_root_stateful,
                             user_id=USER_ID_STATEFUL,
                             session_id=SESSION_ID_STATEFUL
                            )

  # Execute the conversation
  await run_stateful_conversation()

  # Inspect final session state after the conversation
  print("\n--- Inspecting Final Session State ---")
  final_session = session_service_stateful.get_session(app_name=APP_NAME,
                                                       user_id= USER_ID_STATEFUL,
                                                       session_id=SESSION_ID_STATEFUL)
  if final_session:
      print(f"Final Preference: {final_session.state.get('user_preference_temperature_unit')}")
      print(f"Final Last Weather Report (from output_key): {final_session.state.get('last_weather_report')}")
      print(f"Final Last City Checked (by tool): {final_session.state.get('last_city_checked_stateful')}")
  else:
      print("\nError: Could not retrieve final session state.")

else:
  print("\nSkipping state test conversation. Stateful root agent runner ('runner_root_stateful') is not available.")

### Before Model Callback

In [22]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai.types import Content,Part
from typing import Optional

In [23]:
def block_keyword_guardrail(callback_context: CallbackContext,llm_request: LlmRequest)->Optional[LlmResponse]:
    agent_name = callback_context.agent_name
    print(f"--- Callback: block_keyword_guardrail running for agent: {agent_name} ---")

    last_user_message_text = ""
    if llm_request.contents:
        for content in reversed(llm_request.contents):
            if content.role=="user" and content.parts:
                if content.parts[0].text:
                    last_user_message_text = content.parts[0].text
                    break
    
    print(f"--- Callback: Inspecting last user message: '{last_user_message_text[:100]}...' ---")

    # Guardrail logic
    keyword_to_block = "BLOCK"
    if keyword_to_block in last_user_message_text.upper():
        callback_context.state["guardrail_block_keyword_triggered"] = True
        print(f"--- Callback: Set state 'guardrail_block_keyword_triggered': True ---")
        return LlmResponse(
            content=Content(
                role="model",
                parts=[Part(text=f"Can not process the request. Blocked keyword {keyword_to_block}")]
            )
        )
    else:
        return None

In [ ]:
greeting_agent = None
try:
    # Use a defined model constant
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_0_FLASH,
        name="greeting_agent", # Keep original name for consistency
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"Could not redefine Greeting agent. Check Model/API Key ({MODEL_GPT_4O}). Error: {e}")

farewell_agent = None
try:
    # Use a defined model constant
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_0_FLASH,
        name="farewell_agent", # Keep original name
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"Could not redefine Farewell agent. Check Model/API Key ({MODEL_GPT_4O}). Error: {e}")


In [ ]:
root_agent_model_guardrail = None
runner_root_model_guardrail = None

try:
    root_agent_model_guardrail = Agent(
        name="weather_agent_v5_model_guardrail",
        model=MODEL_GEMINI_2_0_FLASH,
        description="""Main agent: Handles weather, delegates greetings/farewells, includes input keyword guardrail.""",
        instruction="""
                    You are the main Weather Agent. Provide weather using 'get_weather_stateful'.
                    Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'.
                    Handle only weather requests, greetings, and farewells.
                    """,
        tools=[get_weather_stateful],
        sub_agents=[farewell_agent,greeting_agent],
        before_model_callback=block_keyword_guardrail,
    )

    print(f"Root Agent '{root_agent_model_guardrail.name}' created with before_model_callback.")

    runner_root_model_guardrail = Runner(
        app_name=APP_NAME,
        agent=root_agent_model_guardrail,
        session_service=session_service_stateful
    )
except Exception as e:
    print(f"Error creating root_agent_model_guardrail. Error {e}")

In [ ]:
async def run_guardrail_test_conversation():
    conversation_func = lambda query:call_agent_async(query=query,
                                                      runner=runner_root_model_guardrail,
                                                      user_id=USER_ID_STATEFUL,
                                                      session_id=SESSION_ID_STATEFUL)
    
    await conversation_func("What is the weather in London?")
    await conversation_func("BLOCK the request for weather in Tokyo")
    await conversation_func("Hello again")

await run_guardrail_test_conversation()


In [ ]:
final_session = session_service_stateful.get_session(app_name=APP_NAME,
                                                       user_id=USER_ID_STATEFUL,
                                                       session_id=SESSION_ID_STATEFUL)
if final_session:
    print("\n--- Final Session State (After Guardrail Test) ---")
    print(f"Guardrail Triggered Flag: {final_session.state.get('guardrail_block_keyword_triggered')}")
    print(f"Last Weather Report: {final_session.state.get('last_weather_report')}") # Should be London weather
    print(f"Temperature Unit: {final_session.state.get('user_preference_temperature_unit')}") # Should be Fahrenheit
else:
    print("Error: Could not retrieve final session state.")

# Before Tool Callback

In [28]:
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from typing import Optional,Dict,Any

In [29]:
def block_paris_tool_guardrail(
        tool:BaseTool,
        args:Dict[str,Any],
        tool_context:ToolContext
)->Optional[Dict]:
    tool_name = tool.name
    agent_name = tool_context.agent_name
    print(f"--- Callback: block_paris_tool_guardrail running for tool '{tool_name}' in agent '{agent_name}' ---")
    print(f"--- Callback: Inspecting args: {args} ---")

    # Guardrail logic
    target_tool_name = "get_weather_stateful"
    blocked_city = "paris"

    if tool_name==target_tool_name:
        city_argument = args.get("city","")
        if city_argument and city_argument.lower()==blocked_city:
            tool_context.state["guardrail_tool_block_triggered"] = True
            print(f"--- Callback: Set state 'guardrail_tool_block_triggered': True ---")

            return {
                "status":"error",
                "error_message":f"Policy restriction: Weather checks for '{city_argument.capitalize()}' are currently disabled by a tool guardrail."
            }
        else:
            print(f"--- Callback: City '{city_argument}' is allowed for tool '{tool_name}'. ---")
    
    return None



In [ ]:
greeting_agent = None
try:
    # Use a defined model constant
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_0_FLASH,
        name="greeting_agent", # Keep original name for consistency
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"Could not redefine Greeting agent. Check Model/API Key ({MODEL_GPT_4O}). Error: {e}")

farewell_agent = None
try:
    # Use a defined model constant
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_0_FLASH,
        name="farewell_agent", # Keep original name
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"Could not redefine Farewell agent. Check Model/API Key ({MODEL_GPT_4O}). Error: {e}")


In [ ]:
root_agent_model_guardrail = None
runner_root_model_guardrail = None

try:
    root_agent_model_guardrail = Agent(
        name="weather_agent_v5_model_guardrail",
        model=MODEL_GEMINI_2_0_FLASH,
        description="""Main agent: Handles weather, delegates greetings/farewells, includes input keyword guardrail.""",
        instruction="""
                    You are the main Weather Agent. Provide weather using 'get_weather_stateful'.
                    Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'.
                    Handle only weather requests, greetings, and farewells.
                    """,
        tools=[get_weather_stateful],
        sub_agents=[farewell_agent,greeting_agent],
        before_model_callback=block_keyword_guardrail,
        before_tool_callback=block_paris_tool_guardrail,
    )

    print(f"Root Agent '{root_agent_model_guardrail.name}' created with before_model_callback.")

    runner_root_model_guardrail = Runner(
        app_name=APP_NAME,
        agent=root_agent_model_guardrail,
        session_service=session_service_stateful
    )
except Exception as e:
    print(f"Error creating root_agent_model_guardrail. Error {e}")

In [ ]:
async def run_guardrail_test_conversation():
    conversation_func = lambda query:call_agent_async(query=query,
                                                      runner=runner_root_model_guardrail,
                                                      user_id=USER_ID_STATEFUL,
                                                      session_id=SESSION_ID_STATEFUL)
    
    await conversation_func("What is the weather in London?")
    await conversation_func("How about Paris?")
    await conversation_func("Hello again")

await run_guardrail_test_conversation()


In [ ]:
final_session = session_service_stateful.get_session(app_name=APP_NAME,
                                                       user_id=USER_ID_STATEFUL,
                                                       session_id=SESSION_ID_STATEFUL)
if final_session:
    print("\n--- Final Session State (After Guardrail Test) ---")
    print(f"Guardrail Triggered Flag: {final_session.state.get('guardrail_block_keyword_triggered',False)}")
    print(f"Tool Guardrail Triggered Flag: {final_session.state.get('guardrail_tool_block_triggered',False)}")
    print(f"Last Weather Report: {final_session.state.get('last_weather_report')}") # Should be London weather
    print(f"Temperature Unit: {final_session.state.get('user_preference_temperature_unit')}") # Should be Fahrenheit
else:
    print("Error: Could not retrieve final session state.")